In [1]:
import SimpleITK as sitk
from downloaddata import fetch_data as fdata
import pydicom
import numpy as np
import os
import matplotlib.pyplot as plt
import itk
import itkwidgets 
import k3d
from k3d.colormaps import matplotlib_color_maps
#import open3d as o3d
import pandas as pd

In [2]:
path="../../DATA/registration_test/FB_100030____FB,2817673489/study_0c1e55cc/"
path_subfolder1=path+"MR7_bb3b1a0b"#"MR3_33a7c6a2"#"MR4_c61744e9"#"MR3_33a7c6a2"
path_subfolder2=path+"MR5_2e0c276a"
path_subfolder3=path+"MR3_33a7c6a2"

In [3]:
# read 2 series (sagittal / coronal / axial)
reader1 = sitk.ImageSeriesReader()
dicom_names1 = reader1.GetGDCMSeriesFileNames(path_subfolder1)
reader1.SetFileNames(dicom_names1)
sag_image = reader1.Execute()
sag_image = sitk.Cast(sag_image, sitk.sitkFloat32)

reader2 = sitk.ImageSeriesReader()
dicom_names2 = reader2.GetGDCMSeriesFileNames(path_subfolder2)
reader2.SetFileNames(dicom_names2)
cor_image = reader2.Execute()
cor_image = sitk.Cast(cor_image, sitk.sitkFloat32)

reader3 = sitk.ImageSeriesReader()
dicom_names3 = reader3.GetGDCMSeriesFileNames(path_subfolder3)
reader3.SetFileNames(dicom_names3)
ax_image = reader3.Execute()
ax_image = sitk.Cast(ax_image, sitk.sitkFloat32)

In [4]:
print('origin: ' + str(sag_image.GetOrigin()))
print('size: ' + str(sag_image.GetSize()))
print('spacing: ' + str(sag_image.GetSpacing()))
print('direction: ' + str(sag_image.GetDirection()))

origin: (117.90852282166, -103.56080762947, 53.280393663713)
size: (320, 320, 32)
spacing: (0.4375, 0.4375, 3.300000171769498)
direction: (0.13914594144969092, 0.03131872101712905, -0.9897765124976096, 0.9902718853820294, -0.0044006872607998656, 0.13907633505939634, -3.951777152039084e-09, -0.9994997607130699, -0.031626386681315836)


In [5]:
print('origin: ' + str(cor_image.GetOrigin()))
print('size: ' + str(cor_image.GetSize()))
print('spacing: ' + str(cor_image.GetSpacing()))
print('direction: ' + str(cor_image.GetDirection()))

origin: (20.955185829075, -102.72428033114, 48.847612927792)
size: (320, 320, 34)
spacing: (0.4375, 0.4375, 3.299999994280998)
direction: (0.9766941032217903, 0.019466772699350238, -0.21375142921733836, 0.2128145932525399, 0.041634528665580504, 0.9762051602616897, 0.027903003969805152, -0.9989432470283119, 0.0365213853241392)


In [6]:
def dcm_reshape(direction, orientation):
    
    npdir = np.zeros([3,3])
    if orientation=='sag':
        npdir[0][0] = direction[0]
        npdir[0][1] = direction[1]
        npdir[0][2] = direction[2]
        npdir[1][0] = direction[3]
        npdir[1][1] = direction[4]
        npdir[1][2] = direction[5]
        npdir[2][0] = direction[6]
        npdir[2][1] = direction[7]
        npdir[2][2] = direction[8]
    elif orientation=='cor':
        npdir[0][0] = direction[6]
        npdir[0][1] = direction[7]
        npdir[0][2] = direction[8]
        npdir[1][0] = direction[3]
        npdir[1][1] = direction[4]
        npdir[1][2] = direction[5]
        npdir[2][0] = -direction[0]
        npdir[2][1] = -direction[1]
        npdir[2][2] = -direction[2]
    else :
        npdir[0][0] = direction[3]
        npdir[0][1] = direction[4]
        npdir[0][2] = direction[5]
        npdir[1][0] = direction[6]
        npdir[1][1] = direction[7]
        npdir[1][2] = direction[8]
        npdir[2][0] = -direction[0]
        npdir[2][1] = -direction[1]
        npdir[2][2] = -direction[2]
        
    return npdir

In [7]:
def spacing_reshape(spacing, orientation):
    print(spacing, orientation)
    if orientation=='sag':
        s=np.array([[spacing[0], 0,0],
                    [0, spacing[1],0], 
                    [0,0, spacing[2]]])
    elif orientation=='cor':
        s=np.array([[spacing[2], 0,0],
                    [0, spacing[1],0], 
                    [0,0, spacing[0]]])
    else :
        s=np.array([[spacing[1], 0,0],
                    [0, spacing[2],0], 
                    [0,0, spacing[0]]])
    return s

In [20]:
def get_corner(image):
    #print(image.shape)
    points=np.array([[0,0,0], 
                     [image.GetSize()[0],0,0], 
                     [0, image.GetSize()[1], 0], 
                     [image.GetSize()[0], image.GetSize()[1], 0], 
                     [0,0,image.GetSize()[2]],
                     [image.GetSize()[0],0,image.GetSize()[2]], 
                     [0, image.GetSize()[1], image.GetSize()[2]], 
                     [image.GetSize()[0], image.GetSize()[1], image.GetSize()[2]]]).astype(np.float32)
    print(points)
    return points

In [9]:
def calcul_physicsCoo(ori, p, A):
    phyCoo=[]
    for i in range(p.shape[0]):
        p_c=ori+np.dot(A, np.transpose(p)[:,i])
        phyCoo.append(p_c)
    phyCoo=np.array(phyCoo)
    return phyCoo

In [ ]:
# rotate volume 
m_sag=sitk.GetArrayFromImage(cor_image)
m_cor=np.rot90(sitk.GetArrayFromImage(cor_image), 1, (0,2))
m_ax= np.rot90(sitk.GetArrayFromImage(ax_image), 1, (0,1))
m_ax=np.rot90(m_ax, 1, (0,2))
print(m_sag.shape, m_cor.shape, m_ax.shape)

In [21]:
# get indexes of the 8corners of each volume
x_sag=get_corner(sag_image)#sitk.GetArrayFromImage(sag_image))
x_cor=get_corner(cor_image)#sitk.GetArrayFromImage(cor_image))
x_ax=get_corner(ax_image)#sitk.GetArrayFromImage(ax_image))

[[  0.   0.   0.]
 [320.   0.   0.]
 [  0. 320.   0.]
 [320. 320.   0.]
 [  0.   0.  32.]
 [320.   0.  32.]
 [  0. 320.  32.]
 [320. 320.  32.]]
[[  0.   0.   0.]
 [320.   0.   0.]
 [  0. 320.   0.]
 [320. 320.   0.]
 [  0.   0.  34.]
 [320.   0.  34.]
 [  0. 320.  34.]
 [320. 320.  34.]]
[[  0.   0.   0.]
 [320.   0.   0.]
 [  0. 320.   0.]
 [320. 320.   0.]
 [  0.   0.  31.]
 [320.   0.  31.]
 [  0. 320.  31.]
 [320. 320.  31.]]


In [11]:
# get spacing of each volume (dependent of orientation)
npspacing_sag=spacing_reshape(sag_image.GetSpacing(),'sag').astype(np.float32)
print(npspacing_sag)
npspacing_cor=spacing_reshape(cor_image.GetSpacing(),'sag').astype(np.float32)
npspacing_ax=spacing_reshape(ax_image.GetSpacing(),'sag').astype(np.float32)
print(npspacing_sag, npspacing_cor, npspacing_ax)

(0.4375, 0.4375, 3.300000171769498) sag
[[0.4375    0.        0.       ]
 [0.        0.4375    0.       ]
 [0.        0.        3.3000002]]
(0.4375, 0.4375, 3.299999994280998) sag
(0.4375, 0.4375, 3.5999999256962862) sag
[[0.4375    0.        0.       ]
 [0.        0.4375    0.       ]
 [0.        0.        3.3000002]] [[0.4375 0.     0.    ]
 [0.     0.4375 0.    ]
 [0.     0.     3.3   ]] [[0.4375 0.     0.    ]
 [0.     0.4375 0.    ]
 [0.     0.     3.6   ]]


In [12]:
# get direction cosin matrix of each volume (dependent of orientation)
dir_sag = dcm_reshape(sag_image.GetDirection(), 'sag')
dir_cor = dcm_reshape(cor_image.GetDirection(), 'sag')
dir_ax = dcm_reshape(ax_image.GetDirection(), 'sag')

In [13]:
A_sag=np.dot(dir_sag, npspacing_sag)
A_cor=np.dot(dir_cor, npspacing_cor)
A_ax=np.dot(dir_ax, npspacing_ax)

In [14]:
origin_sag= sag_image.GetOrigin()
origin_cor= cor_image.GetOrigin()
origin_ax = ax_image.GetOrigin()
print(origin_cor, origin_ax, origin_sag)

(20.955185829075, -102.72428033114, 48.847612927792) (29.439342535285, -105.36338196744, -81.239008678502) (117.90852282166, -103.56080762947, 53.280393663713)


In [15]:
xspa_sag=calcul_physicsCoo(np.array(origin_sag).astype(np.float32), x_sag, A_sag)
print(xspa_sag)

xspa_cor=calcul_physicsCoo(np.array(origin_cor).astype(np.float32), x_cor, A_cor)
print(xspa_cor)

xspa_ax=calcul_physicsCoo(np.array(origin_ax).astype(np.float32), x_ax, A_ax)
print(xspa_ax)

[[ 117.90852356 -103.56080627   53.28039551]
 [ 119.85656674  -89.69699988   53.28039545]
 [ 122.2931445  -104.17690249  -86.64957099]
 [ 124.24118768  -90.3130961   -86.64957105]
 [-927.29553405   43.30381204   19.88292924]
 [-925.34749087   57.16761843   19.88292919]
 [-922.91091311   42.68771582 -120.04703726]
 [-920.96286993   56.55152222 -120.04703731]]
[[  20.95518494 -102.72428131   48.84761429]
 [  35.48350972  -99.55866424   49.26267147]
 [  23.68053311  -96.8954473   -91.0044403 ]
 [  38.2088579   -93.72983022  -90.58938311]
 [-204.76632106  928.14835303   87.41419663]
 [-190.23799627  931.3139701    87.82925382]
 [-202.04097288  933.97718704  -52.43785795]
 [-187.51264809  937.14280412  -52.02280077]]
[[  29.4393425  -105.36338043  -81.23900604]
 [  42.58492409 -102.17593378  -80.25144614]
 [  -3.89087197   30.51549122  -76.13713941]
 [   9.25470962   33.70293787  -75.1495795 ]
 [ -42.10848494 -166.02430886 1066.93567136]
 [ -28.96290335 -162.8368622  1067.92323127]
 [ -75.4

In [ ]:
# get the nearest origin point from the reference origin point
o_sag= xspa_sag[0]
o_cor= xspa_cor[1]
o_ax = xspa_ax[5]
print(o_cor, o_ax, o_sag)

In [ ]:
# set the rotation matrix using a volume orientation as reference
o_dir_sag = dcm_reshape(sag_image.GetDirection(), 'sag')
o_dir_cor = dcm_reshape(cor_image.GetDirection(), 'cor')
o_dir_ax = dcm_reshape(ax_image.GetDirection(), 'ax')
print(o_dir_sag, o_dir_cor, o_dir_ax)

In [ ]:
# set the spacing matrix using a volume orientation as reference (respecting the rotation matrix changes)
o_npspacing_sag=spacing_reshape(sag_image.GetSpacing(),'sag').astype(np.float32)

o_npspacing_cor=spacing_reshape(cor_image.GetSpacing(),'cor').astype(np.float32)
o_npspacing_ax=spacing_reshape(ax_image.GetSpacing(),'ax').astype(np.float32)
print(o_npspacing_sag, o_npspacing_cor, o_npspacing_ax)

In [ ]:
o_A_sag=np.dot(o_dir_sag, o_npspacing_sag)
o_A_cor=np.dot(o_dir_cor, o_npspacing_cor)
o_A_ax=np.dot(o_dir_ax, o_npspacing_ax)

In [ ]:
# get indexes of the 8corners of each volume
o_x_sag=get_corner(sitk.GetArrayFromImage(sag_image))
o_x_cor=get_corner(m_cor)
o_x_ax=get_corner(m_ax)

In [ ]:
xP_sag=calcul_physicsCoo(np.array(o_sag).astype(np.float32), o_x_sag, o_A_sag)
print(xspa_sag)

xP_cor=calcul_physicsCoo(np.array(o_cor).astype(np.float32), o_x_cor, o_A_cor)
print(xspa_cor)

xP_ax=calcul_physicsCoo(np.array(o_ax).astype(np.float32), o_x_ax, o_A_ax)
print(xspa_ax)

In [ ]:
from itkwidgets import view

def itk_view_from_simpleitk(image, sp, direction, orientation, origin):
    """Get a view of an ITK image from a SimpleITK image."""
    
    #np_view = sitk.GetArrayViewFromImage(image)
    print(image.shape)
    itk_view = itk.image_view_from_array(image)
    #sp= spacing_reshape(image.GetSpacing(),orientation)
    print(cor_image.GetSize(), origin, direction, orientation, sp.diagonal())
    itk_view.SetSpacing(sp.diagonal())
    itk_view.SetOrigin(origin)
    
    #itkspacing = itk.matrix_from_array(sp)
    
    itkdir = itk.matrix_from_array(direction)
    
    itk_view.SetDirection(direction)

    return itk_view

In [ ]:
itk_view = itk_view_from_simpleitk(m_cor, spacing_reshape(cor_image.GetSpacing(),'cor'), o_dir_cor, 'cor', o_cor)
itkwidgets.view(itk_view)

In [ ]:
#o = np.array([origin_ref,origin_ref,origin_ref,origin_img, origin_img,origin_img]).astype(np.float32)

#v = np.array([dir_sag[:,0],dir_sag[:,1], dir_sag[:,2],dir_cor[:,0],dir_cor[:,1], dir_cor[:,2]]).astype(np.float32)

#plt_vectors = k3d.vectors(origins=o,vectors=v,colors=[0xff0000,0xff0000,0xec69b6,0xec69b6,0x5dee1f,0x5dee1f,0xe8776a,0xe8776a,0x96065b,0x96065b,0x396b24, 0x396b24])
plt_points = k3d.points(positions=xspa_sag.astype(np.float32),
                        point_size=2.7,
                        shader='3d',
                        color=0x3f6bc5) #, 0x6a329f, 0xc90076, 0xead1dc, 0xd5a6bd, 0xc27ba0, 0xa64d79, 0x741b47])
plt_points_cor = k3d.points(positions=xspa_cor.astype(np.float32),
                        point_size=2.7,
                        shader='3d',
                        color=0xde49a1)
plt_points_ax = k3d.points(positions=xspa_ax.astype(np.float32),
                        point_size=2.7,
                        shader='3d',
                        color=0xead1dc)
#plt_points_origins = k3d.points(positions=np.array([o_cor, o_ax, o_sag]).astype(np.float32),
                        #point_size=4,
                        #shader='3d',
                        #color=0xa64d79)
plt_points_ori = k3d.points(positions=np.array([origin_cor, origin_ax, origin_sag]).astype(np.float32),
                        point_size=4,
                        shader='3d',
                        color=0xc27ba0)
plot = k3d.plot()
#plot += plt_vectors
plot += plt_points
plot += plt_points_cor
plot += plt_points_ax
#plot += plt_points_origins
plot += plt_points_ori
plot.display()

In [ ]:
from itkwidgets import view

def itk_view_from_simpleitk(image, orientation, origin):
    """Get a view of an ITK image from a SimpleITK image."""
    origin = image.GetOrigin()
    spacing = image.GetSpacing()
    direction = image.GetDirection()
    print(origin, spacing, direction)
    np_view = sitk.GetArrayViewFromImage(image)
    itk_view = itk.image_view_from_array(np_view)

    #origin_ref = np.array(ref.GetOrigin())
    origin_img = np.array(origin)
    #T1 = origin_ref-origin_img
    #itk_view.SetOrigin(origin_img +T1)
    itk_view.SetOrigin(origin_img)
    
    npdir_img = dcm_reshape(direction,orientation)
    npspacing = spacing_reshape(spacing, orientation)
    
    itkspacing = itk.matrix_from_array(npspacing)
    #npdir_ref = dcm_reshape(ref.GetDirection())
    
    #inv_dir= np.linalg.inv(npdir_ref)
    #dir2_dir1= np.dot(inv_dir, npdir_img)
    itkdir = itk.matrix_from_array(npdir_img)
    #print(itkdir, npdir_img, npdir_ref)
    itk_view.SetSpacing(itkspacing)
    itk_view.SetDirection(itkdir)

    return itk_view

In [ ]:
A_sag = np.concatenate([dir_sag, np.array([[sag_image.GetOrigin()[0]], [sag_image.GetOrigin()[1]], [sag_image.GetOrigin()[2]]])], axis=1)
A_sag = np.append(A_sag, np.array([[0,0,0,1]]), axis=0)
   
x_sag=np.append(xspa_sag, np.ones((xspa_sag.shape[0],1)), axis=1)
print(xspa_sag)

points_list = []
for i in range(x_sag.shape[0]):
    point=np.dot(A_sag, np.transpose(x_sag)[:,i])
    #print(point)
    points_list.append(point)
points= np.array(points_list)
points= np.delete(points, 3, axis=1)
print(points)

In [ ]:
A_sag = np.concatenate([dir_sag, np.array([[origin_ref[0]], [origin_ref[1]], [origin_ref[2]]])], axis=1)
A_sag = np.append(A_sag, np.array([[0,0,0,1]]), axis=0)
   
x_sag=np.append(xspa_sag, np.ones((xspa_sag.shape[0],1)), axis=1)
print(xspa_sag)
#print(np.transpose(xspa_sag)[:,0])
points_list = []
for i in range(x_sag.shape[0]):
    point=np.dot(A_sag, np.transpose(x_sag)[:,i])
    #print(point)
    points_list.append(point)
points= np.array(points_list)
points= np.delete(points, 3, axis=1)
print(points)

In [ ]:
inv_dir= np.linalg.inv(A_sag)
print(inv_dir)

In [ ]:
#DirSpac_cor= np.multiply(dir2_dir1, s_cor)
A_cor = np.concatenate([dir_cor, np.array([[origin_img[0]], [origin_img[1]], [origin_img[2]]])], axis=1)
A_cor = np.append(A_cor, np.array([[0,0,0,1]]), axis=0)

#A = np.dot(inv_dir, A_cor)
#print(A)

In [ ]:
A_cor = np.concatenate([dir_cor, np.array([[cor_image.GetOrigin()[0]], [cor_image.GetOrigin()[1]], [cor_image.GetOrigin()[2]]])], axis=1)
A_cor = np.append(A_cor, np.array([[0,0,0,1]]), axis=0)

x_cor=np.append(xspa_cor, np.ones((xspa_cor.shape[0],1)), axis=1)
print(x_cor)
points_list_cor = []
for i in range(x_cor.shape[0]):
    point=np.dot(A_cor, np.transpose(x_cor)[:,i])
    points_list_cor.append(point)
points_cor= np.array(points_list_cor)
points_cor= np.delete(points_cor, 3, axis=1)
print(points_cor)

In [ ]:
direction1 = sag_image.GetDirection()
direction2 = cor_image.GetDirection()
origin1 = sag_image.GetOrigin()
origin2 = cor_image.GetOrigin()
spacing1 = sag_image.GetSpacing()
spacing2 = cor_image.GetSpacing()

In [ ]:
print(np.array(origin1), np.array(origin2))

In [ ]:
origin1 = np.array(origin1)
origin2 = np.array(origin2)
T1 = origin1-origin2

In [ ]:
print(T1)
print(origin2+T1)

In [ ]:
direction1=np.reshape(direction1,(3,3))
direction2=np.reshape(direction2,(3,3))
print(np.linalg.det(direction1))
inv_dir1= np.linalg.inv(direction1)
dir2_dir1= np.dot(inv_dir1, direction2)
print(np.dot(inv_dir1, direction1))
print(direction1) 
print(direction2)
print(inv_dir1)
print(dir2_dir1)

In [ ]:
vol1 = sitk.GetArrayFromImage(sag_image)
vol2 = sitk.GetArrayFromImage(cor_image)

In [ ]:
def rotation_matrix(dir1, dir2):
    # Calculate the cross product of dir1 and dir2
    rot_axis = np.cross(dir2, dir1)
    print(rot_axis.shape)
    # Calculate the dot product of dir1 and dir2
    dot_prod = np.dot(dir2, dir1)
    print(dot_prod.shape)
    # Calculate the angle between dir1 and dir2
    angle = np.arcsin(dot_prod / (np.linalg.norm(dir2) * np.linalg.norm(dir1)))
    print(angle.shape)
    # Use the Rodrigues' rotation formula to get the rotation matrix
    k = rot_axis / np.linalg.norm(rot_axis)
    print('here', k.shape)
    c = np.cos(angle)
    print(c.shape)
    s = np.sin(angle)
    print(s.shape)
    v = 1 - c
    print(v.shape)
    R = np.array([[v*k[0]*k[0]+c,  v*k[0]*k[1]-s*k[2], v*k[0]*k[2]+s*k[1]],
                  [v*k[0]*k[1]+s*k[2], v*k[1]*k[1]+c,  v*k[1]*k[2]-s*k[0]],
                  [v*k[0]*k[2]-s*k[1], v*k[1]*k[2]+s*k[0], v*k[2]*k[2]+c]])
    return R

In [ ]:
def rotation_matrix(dir1, dir2):
    r_axis = np.cross(dir1, dir2)
    r_axis /= np.linalg.norm(r_axis)
    
    sin_theta = r_axis / (np.linalg.norm(dir1) * np.linalg.norm(dir2))
    cos_theta = np.dot(dir1, dir2) / (np.linalg.norm(dir1) * np.linalg.norm(dir2))
    
    r_x = np.array([[1.,0.,0.],
                    [0., cos_theta, -sin_theta], 
                    [0., sin_theta, cos_theta]])
    
    r_y = np.array([[cos_theta, 0., sin_theta], 
                    [0.,1.,0.], 
                    [-sin_theta, 0., cos_theta]])
    
    r_z = np.array([[cos_theta, -sin_theta, 0.], 
                    [sin_theta, cos_theta, 0.], 
                    [0.,0., 1.]])
    
    rotation = np.matmul(r_x,r_y)
    R = np.matmul(rotation, r_z)
    #r_axis_cross_sq = np.dot(r_axis_cross, r_axis_cross)
    #R = cos_theta*np.eye(3) + sin_theta*r_axis_cross + (1-cos_theta)*r_axis_cross_sq
    return R

In [ ]:
rotation = rotation_matrix(direction1,direction2)
print(rotation.shape)

In [ ]:
print(spacing2)

In [ ]:
volcoo=[]
for z in range(vol2.shape[0]):
    for x in range(vol2.shape[1]):
        for y in range(vol2.shape[2]):
            i= vol2[z,x,y]
            posx=x*spacing2[0]
            posy=y*spacing2[1]
            posz=z*spacing2[2]
            volcoo.append([posx,posy,posz,i, i, i])
            
vol_cor_coo= pd.DataFrame(volcoo, columns=('x', 'y', 'z', 'R', 'G', 'B'))

In [ ]:
#specify path for export
path = r'vol_cor_coordinates.txt'

#export DataFrame to text file
with open(path, 'a') as f:
    vol_sag_coo_txt = vol_sag_coo.to_string(header=False, index=False)
    f.write(vol_sag_coo_txt)

In [ ]:
pcd = o3d.io.read_point_cloud("vol_sag_coordinates.txt", format='xyzrgb')

In [ ]:
pcd_cor = o3d.io.read_point_cloud("vol_cor_coordinates.txt", format='xyzrgb')

In [ ]:
print(pcd)
print(pcd_cor)

In [ ]:
o3d.visualization.draw_geometries([pcd], zoom=0.2412,
                                  front=[10.4, 10.5, 10.5],
                                  lookat=[0.6172, 0.0475, 2.532],
                                  up=[-0.0694, -0.9768, -0.2024])

In [ ]:
def rotate_view(vis):
        ctr = vis.get_view_control()
        ctr.rotate(10.0, 0.0)
        return False

o3d.visualization.draw_geometries_with_animation_callback([pcd_cor],rotate_view)

In [ ]:
vol1 = sitk.GetArrayFromImage(sag_image)

# define voxel size in different axes
rx=sag_image.GetSpacing()[0]
ry=sag_image.GetSpacing()[2]
rz=sag_image.GetSpacing()[1]

origin=sag_image.GetOrigin()

In [ ]:
z1, y1, x1 = np.meshgrid(np.arange(vol1.shape[0]) * rx + origin[0],
                         np.arange(vol1.shape[1]) * ry + origin[1],
                         np.arange(vol1.shape[2]) * rz + origin[2],
                         indexing='ij')
print(np.size(x1))
XYZ=np.zeros((np.size(x1),3))
XYZ[:,0] = np.reshape(x1, -1)
XYZ[:,1] = np.reshape(y1, -1)
XYZ[:,2] = np.reshape(z1, -1)

print(XYZ.shape)

In [ ]:
# Pass xyz to Open3D.o3d.geometry.PointCloud and visualize
pcd = o3d.geometry.PointCloud()

In [ ]:
pcd.points = o3d.utility.Vector3dVector(XYZ)
o3d.io.write_point_cloud("voltest.ply", pcd)

In [ ]:
o3d.visualization.draw_geometries([pcd])

In [ ]:
# Generate some neat n times 3 matrix using a variant of sync function
x = np.linspace(-3, 3, 401)
print(x.shape)
mesh_x, mesh_y = np.meshgrid(x, x)
print(np.size(mesh_x))
z = np.sinc((np.power(mesh_x, 2) + np.power(mesh_y, 2)))
z_norm = (z - z.min()) / (z.max() - z.min())
xyz = np.zeros((np.size(mesh_x), 3))
print(xyz.shape)
xyz[:, 0] = np.reshape(mesh_x, -1)
xyz[:, 1] = np.reshape(mesh_y, -1)
xyz[:, 2] = np.reshape(z_norm, -1)
print('xyz')
print(xyz)

In [ ]:
# Pass xyz to Open3D.o3d.geometry.PointCloud and visualize
pcd = o3d.geometry.PointCloud()
pcd.points = o3d.utility.Vector3dVector(xyz)
o3d.io.write_point_cloud("sync.ply", pcd)

In [ ]:
# Load saved point cloud and visualize it
pcd_load = o3d.io.read_point_cloud("sync.ply")



In [ ]:
# Convert Open3D.o3d.geometry.PointCloud to numpy array
xyz_load = np.asarray(pcd_load.points)


In [ ]:
print('xyz_load')
print(xyz_load)


In [ ]:
o3d.visualization.draw_geometries([pcd_load])